# CEG 论文结果包 Colab Pipeline

该 Notebook 用于在图像生成产物已经存在后, 调用仓库脚本完成 attack、真实 detection、fixed-FPR 校准、论文结果包导出和 Google Drive 归档。方法逻辑不写在 Notebook 中。


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_URL = 'https://github.com/RICHAAARC/CEG.git'
REPO_BRANCH = ''
REPO_ROOT = Path('/content/CEG')
DRIVE_ROOT = Path('/content/drive/MyDrive/CEG')
DRIVE_INPUT_WORKSPACE_ROOT = DRIVE_ROOT / 'pilot_runs' / 'real_pilot_input_workspace_20260617_034500'
LOCAL_RUNTIME_ROOT = Path('/content/ceg_runtime')
WORKSPACE = LOCAL_RUNTIME_ROOT / DRIVE_INPUT_WORKSPACE_ROOT.name
RESET_LOCAL_RUNTIME_WORKSPACE = True

RUN_PIPELINE = True
ALLOW_INCOMPLETE_PACKAGE = True
ALLOW_INVALID_ARCHIVE = True
ATTACK_FAMILIES = 'brightness_contrast,gaussian_noise,rotate,resize,jpeg'
PERSPECTIVE_OFFSETS = '0.0,0.04,-0.04'
LOCAL_DEFORMATION_ENABLED = 'true'
TARGET_FPR = 0.01
ATTESTATION_KEY_ENV = 'CEG_ATTESTATION_KEY'
ATTESTATION_KEY_ID = 'formal_colab_run_key'
SEMANTIC_MASK_BACKEND = 'ceg_inspyrenet_semantic_mask'
PREPARE_INSPYRENET_WEIGHT = True
INSPYRENET_WEIGHT_URL = 'https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth'
INSPYRENET_WEIGHT_DRIVE_PATH = DRIVE_ROOT / 'Models' / 'inspyrenet' / 'ckpt_base.pth'
INSPYRENET_CACHE_PATHS = [Path.home() / '.transparent-background' / 'ckpt_base.pth', Path.home() / '.cache' / 'transparent-background' / 'ckpt_base.pth']
BASELINE_OBSERVATIONS = WORKSPACE / 'external_baselines' / 'baseline_observations.json'
USE_BASELINE_OBSERVATIONS = BASELINE_OBSERVATIONS.exists()


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'非 Colab 环境或 Drive 已挂载: {exc}')

if not REPO_ROOT.exists():
    cmd = ['git', 'clone']
    if REPO_BRANCH:
        cmd += ['--branch', REPO_BRANCH]
    cmd += [REPO_URL, str(REPO_ROOT)]
    print('运行:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
else:
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--all', '--prune'], check=True)
    if REPO_BRANCH:
        subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_ROOT)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transparent-background'], check=True)


def ensure_ceg_attestation_key() -> None:
    """在 Colab 运行时定义 CEG_ATTESTATION_KEY, 只放入环境变量, 不写入文件或日志。"""
    if os.environ.get(ATTESTATION_KEY_ENV):
        print(f"{ATTESTATION_KEY_ENV} 已存在, 不打印密钥内容。")
        return
    secret_value = None
    try:
        from google.colab import userdata  # type: ignore
        secret_value = userdata.get(ATTESTATION_KEY_ENV)
    except Exception:
        secret_value = None
    if not secret_value:
        import secrets
        secret_value = secrets.token_urlsafe(48)
        print(f"{ATTESTATION_KEY_ENV} 未在 Colab secrets 中定义, 已为本次运行生成临时密钥。")
    os.environ[ATTESTATION_KEY_ENV] = secret_value
    print(f"{ATTESTATION_KEY_ENV} configured =", bool(os.environ.get(ATTESTATION_KEY_ENV)))


def prepare_inspyrenet_weight() -> None:
    """准备 InSPyReNet 权重, 该逻辑仅属于 Colab 环境准备, 不进入 CEG 主方法。"""
    if not PREPARE_INSPYRENET_WEIGHT:
        print("PREPARE_INSPYRENET_WEIGHT = False, 跳过 InSPyReNet 权重准备。")
        return
    source_path = INSPYRENET_WEIGHT_DRIVE_PATH
    if not source_path.is_file():
        source_path.parent.mkdir(parents=True, exist_ok=True)
        download_cmd = ["python", "-c", "import urllib.request, sys; urllib.request.urlretrieve(sys.argv[1], sys.argv[2])", INSPYRENET_WEIGHT_URL, str(source_path)]
        print("Drive 中未找到 InSPyReNet 权重, 下载到:", source_path)
        subprocess.run(download_cmd, check=True)
    for cache_path in INSPYRENET_CACHE_PATHS:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        if not cache_path.exists():
            shutil.copyfile(source_path, cache_path)
    os.environ["INSPYRENET_CKPT_PATH"] = str(source_path)
    print("InSPyReNet 权重已准备, 不写入主方法代码:", source_path)

ensure_ceg_attestation_key()
prepare_inspyrenet_weight()




def prepare_local_runtime_workspace() -> None:
    """从 Google Drive 读取前序产物到 Colab 本地运行目录。"""
    if not DRIVE_INPUT_WORKSPACE_ROOT.exists():
        raise FileNotFoundError(f'Google Drive 输入工作区不存在: {DRIVE_INPUT_WORKSPACE_ROOT}')
    LOCAL_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
    if WORKSPACE.exists() and RESET_LOCAL_RUNTIME_WORKSPACE:
        shutil.rmtree(WORKSPACE)
    if not WORKSPACE.exists():
        shutil.copytree(DRIVE_INPUT_WORKSPACE_ROOT, WORKSPACE)
    print('已把 Drive 前序产物复制到 Colab 本地运行工作区:', WORKSPACE)

prepare_local_runtime_workspace()


In [ ]:
cmd = [
    sys.executable,
    'scripts/run_colab_paper_results_pipeline.py',
    '--workspace', str(WORKSPACE),
    '--drive-root', str(DRIVE_ROOT),
    '--attack-families', ATTACK_FAMILIES,
    '--target-fpr', str(TARGET_FPR),
    '--attestation-key-env', ATTESTATION_KEY_ENV,
    '--attestation-key-id', ATTESTATION_KEY_ID,
    '--semantic-mask-backend', SEMANTIC_MASK_BACKEND,
    '--detection-formal-result-claim',
    '--perspective-offsets', PERSPECTIVE_OFFSETS,
    '--local-deformation-enabled', LOCAL_DEFORMATION_ENABLED,
]
if USE_BASELINE_OBSERVATIONS:
    cmd += ['--baseline-observations', str(BASELINE_OBSERVATIONS)]
if ALLOW_INCOMPLETE_PACKAGE:
    cmd.append('--allow-incomplete-package')
if ALLOW_INVALID_ARCHIVE:
    cmd.append('--allow-invalid-archive')
print('运行:', ' '.join(cmd))
if RUN_PIPELINE:
    subprocess.run(cmd, check=True)
else:
    print('RUN_PIPELINE = False, 仅打印命令。')


In [ ]:
paths = {
    'drive_input_workspace': DRIVE_INPUT_WORKSPACE_ROOT,
    'local_runtime_workspace': WORKSPACE,
    'pipeline_manifest': WORKSPACE / 'paper_results_pipeline' / 'colab_paper_results_pipeline_manifest.json',
    'paper_results_package': WORKSPACE / 'paper_results_pipeline' / 'calibrated_paper_results_package' / 'paper_results_package',
    'drive_archives': DRIVE_ROOT / 'package_archives',
}
for name, path in paths.items():
    print(name, '=', path, 'exists=', path.exists())
